<a href="https://colab.research.google.com/github/olgyan/irma/blob/try/cts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
from IPython.display import display, clear_output, HTML
from collections import namedtuple, defaultdict
from shutil import rmtree
from subprocess import run
from tqdm.auto import tqdm

import io
import ipywidgets as widgets
import os
import pandas as pd
import re

tqdm.pandas()
workdir = '/content/drive/MyDrive'

In [35]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Функции
Их можно свернуть

Антикапслок

In [36]:
# @title
def uncapslock(phrase):
    """Like built-in capitalize() but for multi-sentence texts:
    keep uppercase letters after periods. Does not work if there are
    digits in the phrase.
    """

    new = phrase.capitalize()
    if '. ' in new:
        new = re.sub(
            r'(?<=\. )(.)',
            lambda m: m.group(1).upper(),
            new)
    # if new != phrase:
    #     print(f'{phrase} > {new}')

    # return Roman numbers to uppercase
    new = re.sub(
        r'((^|\b)[IVXivx]+($|\b))',
        lambda m: m.group(1).upper(),
        new)
    return new

Кэширование

In [37]:
# @title
# @Кэширование оглавлений в текстовые файлы (название файла - шифр книги)
def make_caches(row):
    """
    Args: row of the books dataframe
    Caches tables of contents in .txt files (for backup)
    """
    cache_file = f'{workdir}/txt_cache/{row.db_id}.txt'
    if not os.path.exists(cache_file):
        with open(cache_file, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(str(row.toc))
    return row

In [78]:
# These are regexes to search for personal names of authors
personal_name = (
    re.compile(r'((?:[А-Я]\. +)+[А-Я][а-я]+)'),
    re.compile(r'([А-Я][а-я]+(?:[А-Я]\. +)+)'),
    re.compile(r'([А-Я])([а-я]+) ([А-Я][а-я]+)')
)
PAGEREGEX = re.compile(r'(.+?) ?[ _.…\t]{3,} ?(\d+)')

# Don't put these unto the C subfield, put them into U subfield
unneeded_titles = ('Введение', 'Предисловие', 'Заключение', 'Литература',
                   'Контрольные вопросы', 'Библиографический список',
                   'Предметный указатель')

# @Отделяем номера глав или имена авторов
def make_numbering_or_authors(row):
    """
    Takes a row from the f330 DataFrame, returns modified row.
    Splits row.A along '. '. Makes new columns U, C, G (that are Irbis subfields
    for chapter number, chapter title and resp.stmt.) and puts the first part
    in 'U' or 'G' (depending on contents type passed to the function or guessed)
    and the second part in 'C'. Authors' names from 'G' will be parsed further.
    """
    if row.A in unneeded_titles:
        # moving all of it to ^U
        row['U'] = row.A
        row.A = None
        in_parts = ('', '', '')
    else:
        # splitting chapter
        in_parts = row.A.partition('. ')

    contents_type = 'C'

    if in_parts[1] == '. ':
        # split successful
        # automatic deciding logic based on what the first part looks like
        if any((
            row.A.startswith('Глава'),
            row.A.startswith('Раздел'),
            not in_parts[0].isalpha()
        )):
            contents_type = 'U'
        elif any(re.match(regex, row.A) for regex in (personal_name)):
            contents_type = 'G'

    # must add logic for manual contents_type selection later
    # CONTENTS_TYPES = {'имена авторов': 'G', 'номера глав': 'U', 'ничего': 'C'}

    print(f'{contents_type=}')
    # putting parts in subfields
    row[contents_type] = in_parts[0]
    if contents_type != 'C':
        row.C = in_parts[2]

    if row.C.isupper():
        row.C = uncapslock(row.C)

    return row

# @Отделяем страницы
def make_title_and_pages(toc):
    """
    Accepts tables of contents in strings.
    Separates scanned table of contents into chapter names and page numbers.
    Makes ranges from page numbers. Puts results into a pandas.DataFrame f330
    and returns it.
    """
    f330 = pd.DataFrame(PAGEREGEX.findall(toc), columns=['A', 'B'])

    # insert consistent page separators for pageregex to work
    toc = toc.replace('…', '...')
    toc = toc.replace(' .', '.')
    while '  ' in toc:
        toc = toc.replace('  ', ' ')

    if '\n' in f330.A[0]:
        f330.A[0] = f330.A[0].partition('\n')[2]
    f330.A = f330.A.str.strip()
    f330.B = f330.B.astype('int64')

    def pagerange(row: pd.Series):
        nonlocal f330
        page = row.B
        lastpage = f330.B[row.i+1] - 1 if (row.i + 1) < f330.shape[0] else page
        pagerange = f'{page}-{lastpage}' if lastpage > page else str(page)
        return pagerange

    f330['i'] = f330.index
    f330['4'] = f330.apply(pagerange, axis=1)
    f330 = f330.drop(columns=['i', 'B'])

    # applying make_numbering_or_authors
    # f330 = f330.apply(make_numbering_or_authors, axis=1)
    # f330 = f330.dropna(axis=1)

    return f330

def books_row_treating(row):
    """
    Applies make_title_and_pages to row.toc, gets the dataframe
    and saves it in pickle file (workdir)/pkl_cache/(db_id).pkl for unpickling
    and continuing parsing later.
    Errors are collected in 'success'.
    """
    try:
        f330 = make_title_and_pages(row.toc)
        f330 = f330.apply(make_numbering_or_authors, axis=1)
        output_file = f'{workdir}/pkl_cache/{row.db_id}.pkl'
        f330.to_pickle(output_file)
        row.success = True

    except Exception as e:
        row.success = str(e)

    finally:
        return row

# testing
with open(f'{workdir}/txt_cache/1225378.txt') as f:
    text = f.read()

f330 = make_title_and_pages(text)
f330 = f330.apply(make_numbering_or_authors, axis=1)
display(f330)

contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='C'
contents_type='U'
contents_type='U'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='U'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_type='C'
contents_t

,4,A,C,U
0,10-13,Предисловие от издательства,Предисловие от издательства,NaN
1,14-17,Особая благодарность,Особая благодарность,NaN
2,18,Замечание о качестве изображений,Замечание о качестве изображений,NaN
3,18,О загружаемом контенте,О загружаемом контенте,NaN
4,19,Глава 1. Работа над «видом»,NaN,Глава 1
...,...,...,...,...
100,231,Метод 2. Постаревшие красители пленки,NaN,Метод 2
101,232-234,Метод 3. Эффектный черно-белый вид,NaN,Метод 3
102,235-244,Метод 4. Эффект тонированных черно-белых пленок,NaN,Метод 4
103,245,None,,Заключение


In [40]:
# @Получили данные в виде датафрейма
# и применяем к ним make_330s_lite и прочие функции
def parse_df(books: pd.DataFrame):
    """
    Args: dataframe
    Try to parse all TOCs from dataframe simultaneously
    Returns: dataframe
    """
    books['db_id'] = books['db_id'].str.replace("/", "_",
        case=False, regex=False) # This makes id's suitable as filenames

    books = books.drop(columns=['Unnamed: 0'])
    books['success'] = False
    books.drop_duplicates(subset=['title', 'authors', 'url'])
    print('Удалили дубликаты. Результат:')
    books.info()

    print('Кэшируем оглавления')
    books = books.progress_apply(make_caches, axis=1)

    print('Разбираем оглавления')
    books = books.progress_apply(make_title_and_pages, axis=1)
    books.info()
    print('Результат:')
    display(books.head(), books.tail())
    return books

Функция, разбирающая текстовый формат Ирбис

In [41]:
# @title
# Получили текстовые данные, делаем датафрейм и вызываем parse_df на него
def parse_irbis(irbis_text):
    """
    Args: string of Irbis text
    Makes dataframe from it
    Returns: dataframe
    """
    if '\n*****\n' in submission_data:
        records = submission_data.split('\n*****\n')
    else:
        records = (submission_data,) # make data an iterable anyway
    display(f'{records=}')

    regex_findings = []
    regexes = {
    'title': lambda record: re.search(r"(?<=#200: \^A).+?(?=(\^|$))", record),
    'authors': lambda record: re.search(
        r"(?<=#200: )\^A.+\^F(.+?)(?=(?:\^|$))", record),
    'toc': lambda record: re.search(r"(?<=#330: )(?:\^A)?(.+?)(?=$)", record),
    'db_id': lambda record: re.search(r"(?<=#903: ).+?(?=$)", record)
    }

    for rec in records:
        """
        Apply regexes to records, get dicts
        """
        row = {}
        for k, v in regexes.items():
            row[k] = (v[1] if k in ('authors', 'toc') else v[0]
                ) if type(v) is re.Match else ''

        display(f'{row=}')

        # delete TOC from Irbis record and save record for later
        row['other_irbis_text'] = rec.replace(row['toc'], '')
        regex_findings.append(row)

    books = parse_df(pd.DataFrame.from_records(regex_findings))
    return books

# Работа

Для начала работы нажимаем Runtime > Run before или Ctrl+F8 (это запускает все ячейки выше этой и загружает функции).
Если "Ирма" ещё не получала исходные данные, здесь можно их ей дать. Исходные данные должны содержать заглавия, шифры и необработанные оглавления книг и быть в формате .csv или текстовом формате Ирбис.

In [ ]:
# @title

# Создаем виджеты
toggle = widgets.ToggleButtons(
    options=['Загрузить файл', 'Открыть файл .csv', 'Ввести текст'],
    description='Метод ввода:',
    style={'description_width': 'initial'}
)

file_upload = widgets.FileUpload(
    description='Выберите файл:',
    multiple=False,
    layout={'display': 'none', 'width': '600px'}
)

text_entry = widgets.Textarea(
    description='Введите оглавление:',
    layout={'display': 'none', 'width': '600px', 'height': '150px'}
)

file_entry = widgets.Text(
    description='Путь к файлу:',
    value=f'{workdir}/backup.csv',
    layout={'display': 'none', 'width': '600px'}
)

submit = widgets.Button(
    description="Отправить",
    button_style='success'
)

output = widgets.Output()
result = widgets.Output()

# Переменная для хранения результатов
submission_data = None

def on_toggle_change(change):
    """Обработчик изменения переключателя"""
    with output:
        output.clear_output()
        file_upload.layout.display = 'none'
        file_entry.layout.display = 'none'
        text_entry.layout.display = 'none'

        if change['new'] == 'Загрузить файл':
            file_upload.layout.display = 'block'
            print("Пожалуйста, загрузите CSV или TXT файл")
        elif change['new'] == 'Открыть файл .csv':
            file_entry.layout.display = 'block'
            print("Пожалуйста, введите путь к файлу")
        else:
            text_entry.layout.display = 'block'
            print("Пожалуйста, введите ваш текст")

def handle_submission(button):
    """Обработчик отправки данных"""
    global submission_data

    with output:
        output.clear_output()

        if toggle.value == 'Загрузить файл' and file_upload.value:
            try:
                uploaded_file = next(iter(file_upload.value.values()))
                filename = uploaded_file['name']
                content = uploaded_file['content']

                if filename.endswith('.csv'):
                    submission_data = pd.read_csv(io.BytesIO(content), sep='\t')
                    with result:
                        result.clear_output()
                        print("═"*50)
                        print(f"ФАЙЛ УСПЕШНО ОБРАБОТАН: {filename}")
                        print("Первые 5 строк:")
                        display(submission_data.head())
                        print("═"*50)
                elif filename.endswith('.txt'):
                    submission_data = content.decode('utf-8')
                    with result:
                        result.clear_output()
                        print("═"*50)
                        print(f"СОДЕРЖИМОЕ TXT ФАЙЛА: {filename}")
                        print(submission_data)
                        print("═"*50)
                else:
                    with result:
                        result.clear_output()
                        print("ОШИБКА: Поддерживаются только файлы .csv и .txt")

            except Exception as e:
                with result:
                    result.clear_output()
                    print("═"*50)
                    print("ОШИБКА ОБРАБОТКИ ФАЙЛА:")
                    print(str(e))
                    print("═"*50)

        elif toggle.value == 'Открыть файл .csv' and file_entry.value:
            try:
                submission_data = pd.read_csv(file_entry.value, sep='\t')
                with result:
                    result.clear_output()
                    print("═"*50)
                    print(f"ФАЙЛ УСПЕШНО ОБРАБОТАН: {file_entry.value}")
                    print("Первые 5 строк:")
                    display(submission_data.head())
                    print("═"*50)
            except Exception as e:
                with result:
                    result.clear_output()
                    print("═"*50)
                    print("ОШИБКА ОТКРЫТИЯ ФАЙЛА:")
                    print(str(e))
                    print("═"*50)

        elif toggle.value == 'Ввести текст' and text_entry.value:
            submission_data = text_entry.value
            with result:
                result.clear_output()
                print("═"*50)
                print("ВВЕДЕННЫЙ ТЕКСТ:")
                print(submission_data)
                print("═"*50)

        else:
            with result:
                result.clear_output()
                print("ОШИБКА: Пожалуйста, сначала введите данные")

# Назначаем обработчики
toggle.observe(on_toggle_change, names='value')
submit.on_click(handle_submission)

# Отображаем все виджеты
print("═══════════════════════════════════════════════")
print("        ИНТЕРАКТИВНЫЙ ИНТЕРФЕЙС ВВОДА")
print("═══════════════════════════════════════════════")
display(toggle, file_upload, file_entry, text_entry, submit)
display(output, result)

Эта ячейка обрабатывает исходные данные: сохраняет оглавления в отдельных текстовых файлах в папке txt_cache, пытается из каждого сделать набор подполей, результаты сохраняет в папке pkl_cache, позднее их можно будет просмотреть.

In [ ]:
# @title
# We got submission_data. Now we apply contents-processing functions

if submission_data is not None:
    print("═══════════════════════════════════════════════")
    print("             ОБРАБОТКА ОГЛАВЛЕНИЙ")
    print("═══════════════════════════════════════════════")

    if isinstance(submission_data, pd.DataFrame):
        books = parse_df(submission_data)

    elif isinstance(submission_data, str):
        irbis_traits = ('#200: ', '#330:', '#903: ', '\n*****\n')
        if all(x in submission_data for x in irbis_traits):
            books = parse_irbis(submission_data)

# Now we have 'books' dataframe

Очистка кэшей текстовых файлов

In [ ]:
rmtree(f'{workdir}/txt_cache')

Очистка кэша результатов разбора

In [66]:
rmtree(f'{workdir}/pkl_cache')

Эта ячейка делает резервную копию записей с книгами (backup.csv), если мы загружали их только что. Иначе она загружает записи из резервной копии в оперативную память.

In [9]:
# Backup books dataframe if it is there in working memory
# or load it from backup if it is not
try:
    books.to_csv(f'{workdir}/backup.csv')
except NameError or UnboundLocalError:
    books = pd.read_csv(f'{workdir}/backup.csv')
    books = parse_df(books)
    books.drop(columns=[col for col in books.columns if 'Unnamed' in col])
books.info()


Удалили дубликаты. Результат:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1506 entries, 0 to 1505
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Unnamed: 0.2  1506 non-null   int64 
 1   Unnamed: 0.1  1506 non-null   int64 
 2   title         1506 non-null   object
 3   authors       1404 non-null   object
 4   toc           1506 non-null   object
 5   url           1506 non-null   object
 6   db_id         1506 non-null   object
 7   success       1506 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 84.0+ KB
Кэшируем оглавления


  0%|          | 0/1506 [00:00<?, ?it/s]

Разбираем оглавления


  0%|          | 0/1506 [00:00<?, ?it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1506 entries, 0 to 1505
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Unnamed: 0.2  1506 non-null   int64 
 1   Unnamed: 0.1  1506 non-null   int64 
 2   title         1506 non-null   object
 3   authors       1404 non-null   object
 4   toc           1506 non-null   object
 5   url           1506 non-null   object
 6   db_id         1506 non-null   object
 7   success       1506 non-null   object
dtypes: int64(2), object(6)
memory usage: 94.3+ KB
Результат:


,Unnamed: 0.2,Unnamed: 0.1,title,authors,toc,url,db_id,success
0,0,0,Физика твердого тела. Электронные свойства тве...,Н. Г. Замкова,Предисловие .....................................,https://znanium.com/catalog/document/?pid=2091...,2091865,True
1,1,1,Информационные системы,О. Л. Голицына,Введение . . . . . . . . . . . . . . . . . . ....,https://znanium.com/catalog/document/?pid=1832...,1832410,True
2,2,2,Цветокоррекция творческие стили для кино и видео,В. Хуркман,Предисловие от издательства .....................,https://znanium.com/catalog/document/?pid=1225...,1225378,True
3,3,3,Кинокостюм. Макет. Моделирование. Реконструкци...,Э. В. Герц,Вступительная статья Стыцюк Н.В. . . . . . . ....,https://znanium.com/catalog/document/?pid=1242...,1242019,True
4,4,4,Кинопроект. Практикум начинающего продюсера,NaN,Предисловие________3 Глава 1. ПРАКТИЧЕСКИЕ П...,https://znanium.com/catalog/document/?pid=1352...,1352979,True


,Unnamed: 0.2,Unnamed: 0.1,title,authors,toc,url,db_id,success
1501,1501,1501,Математика. Линейная алгебра,К. М. Расулов,ПРЕДИСЛОВИЕ......................................,https://znanium.com/catalog/document/?pid=1081...,1081982,True
1502,1502,1502,Электротехника в примерах и задачах,А. Е. Поляков,5 ftFpFpftFpftОГЛАВЛЕНИЕ 6 ГЛАВА 1. ЦЕПИ П...,https://znanium.com/catalog/document/?pid=1657...,1657587,True
1503,1503,1503,"Спортсмен, музыкант, поэт, математик… Как выяв...",Е. Первушина,Предисловие. ....................................,https://znanium.com/catalog/document/?pid=1003...,1003020,True
1504,1504,1504,Охрана труда,Т. С. Иванова,ОБЩИЕ МЕТОДИЧЕСКИЕ РЕКОМЕНДАЦИИ ПО ИЗУЧЕНИЮ Д...,https://znanium.com/catalog/document/?pid=1087...,1087921,True
1505,1505,1505,Основы финансовой грамотности,В. А. Кальней,Авторский коллектив .............................,https://znanium.com/catalog/document/?pid=2090...,2090562,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1506 entries, 0 to 1505
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Unnamed: 0.2  1506 non-null   int64 
 1   Unnamed: 0.1  1506 non-null   int64 
 2   title         1506 non-null   object
 3   authors       1404 non-null   object
 4   toc           1506 non-null   object
 5   url           1506 non-null   object
 6   db_id         1506 non-null   object
 7   success       1506 non-null   object
dtypes: int64(2), object(6)
memory usage: 94.3+ KB


## Просматриваем результаты

Создадим итератор для последовательного просмотра. Он одноразовый - чтобы заново просматривать книги, надо его пересоздать.

In [11]:
# @title
books_iterator = books.itertuples(name='Book')

Теперь запускаем последовательный просмотр. Запускаем ячейку один раз и дальше управляем ей с помощью выведенных кнопок.

In [12]:
# @title
# DeepSeek написал
# Глобальные переменные для хранения состояния
current_book = None
cache_file = None
result_df = None

def show_book_interface():
    """Основная функция отображения интерфейса"""
    global current_book, cache_file, result_df

    clear_output(wait=True)

    try:
        # Получаем следующую книгу
        current_book = next(books_iterator)
        cache_file = f'{workdir}/txt_cache/{current_book.db_id}.txt'

        # Создаем файл если его нет
        if not os.path.exists(cache_file):
            with open(cache_file, 'w', encoding='utf-8') as f:
                f.write(current_book.toc)

        # Показываем информацию о книге
        print(f'📖 Книга: {current_book.authors}. {current_book.title}')
        print(f'🔖 Шифр: {current_book.db_id}')
        if current_book.url:
            display(HTML(
                f'<a href="{current_book.url}" target="_blank">Книга в ЭБС</a>'
                ))
        print('═'*50)

        # Пытаемся загрузить результаты
        try:
            result_df = pd.read_pickle(
                f'{workdir}/pkl_cache/{current_book.db_id}.pkl')
            display(result_df)
        except:
            print("Результаты еще не сгенерированы")
            result_df = None

        # Создаем и отображаем кнопки
        display(create_buttons())

    except StopIteration:
        print("✅ Все книги обработаны!")
        return

def refresh_results():
    """Обновляет результаты без рекурсивного вызова show_book_interface"""
    global result_df

    clear_output(wait=True)

    # Повторно показываем информацию о текущей книге
    print(f'📖 Книга: {current_book.authors}. {current_book.title}')
    print(f'🔖 Шифр: {current_book.db_id}')
    print('═'*50)

    # Загружаем обновленные результаты
    try:
        result_df = pd.read_pickle(f'{workdir}/pkl_cache/{current_book.db_id}.pkl')
        display(result_df)
    except:
        print("Результаты еще не сгенерированы")
        result_df = None

    # Отображаем кнопки
    display(create_buttons())

def create_buttons():
    """Создает интерфейс кнопок"""
    buttons_layout = widgets.Layout(width='300px')

    edit_button = widgets.Button(
        description="✏️ Редактировать текст",
        style={'button_color': 'lightblue'},
        layout=buttons_layout
    )

    apply_button = widgets.Button(
        description="🔄 Применить функцию",
        style={'button_color': 'lightgreen'},
        layout=buttons_layout,
        disabled=(result_df is not None)
    )

    next_button = widgets.Button(
        description="⏭️ Следующая книга",
        style={'button_color': 'lightgray'},
        layout=buttons_layout
    )

    # Обработчики кнопок
    def on_edit_click(b):
        edit_content()

    def on_apply_click(b):
        apply_processing()
        refresh_results()

    def on_next_click(b):
        show_book_interface()

    edit_button.on_click(on_edit_click)
    apply_button.on_click(on_apply_click)
    next_button.on_click(on_next_click)

    return widgets.HBox([edit_button, apply_button, next_button])

def edit_content():
    """Отдельная функция для редактирования содержимого"""
    with open(cache_file, 'r', encoding='utf-8') as f:
        content = f.read()

    text_area = widgets.Textarea(
        value=content,
        layout={'width': '100%', 'height': '300px'}
    )

    save_button = widgets.Button(
        description="💾 Сохранить изменения",
        style={'button_color': 'lightgreen'}
    )

    def on_save_click(b):
        with open(cache_file, 'w', encoding='utf-8') as f:
            f.write(text_area.value)
        print("Изменения сохранены!")
        refresh_results()

    save_button.on_click(on_save_click)

    clear_output(wait=True)
    display(widgets.VBox([
        widgets.HTML("<h3>Редактирование текста:</h3>"),
        text_area,
        save_button,
        widgets.Button(
            description="Назад",
            layout=widgets.Layout(width='300px'),
            button_style='',
            on_click=lambda b: refresh_results()
        )
    ]))

def apply_processing():
    """Отдельная функция для применения обработки"""
    try:
        with open(cache_file, 'r', encoding='utf-8') as f:
            content = f.read()

        # Здесь вызывается ваша функция обработки
        current_book.toc = content
        current_book.apply(make_title_and_pages)
        print("Оглавление обработано заново!")

    except Exception as e:
        print(f"Ошибка при обработке: {str(e)}")

In [15]:
show_book_interface()

📖 Книга: В. Хуркман. Цветокоррекция  творческие стили для кино и видео
🔖 Шифр: 1225378
══════════════════════════════════════════════════


,db_id,4,C
0,1225378,10-11,Предисловие от издательства
1,1225378,12-13,
2,1225378,14-17,Особая благодарность
3,1225378,18,Замечание о качестве изображений
4,1225378,18,О загружаемом контенте
...,...,...,...
101,1225378,231,Постаревшие красители пленки
102,1225378,232-234,Эффектный черно-белый вид
103,1225378,235-244,Эффект тонированных черно-белых пленок
104,1225378,245,


In [ ]:
# @Записываем результаты в текстовый формат Ирбис
